In [ ]:
%%writefile bonus_etudiant.cpp
#include <iostream>
#include <vector>
#include <cmath>
#include <random>
#include <fstream>
#include <string>

using namespace std;

// Paramètres
const int N = 20000;
const int GRILLE = 300;
const int JOURS = 730;

// États
enum { S=0, E=1, I=2, R=3 };

// Structure simple pour un agent
struct Humain {
    int x, y;
    int etat;
    int compteur;
    int dE, dI, dR;
};

// --- PARTIE ALÉATOIRE ---
// On déclare les générateurs en global pour faire simple
mt19937 generateur_mt; // Le générateur standard
unsigned int graine_xor; // La graine pour le générateur rapide

// Initialisation
void init_random(int graine) {
    generateur_mt.seed(graine);
    if(graine == 0) graine_xor = 12345;
    else graine_xor = graine;
}

// Fonction qui renvoie un nombre réel entre 0 et 1
// Si type==0 -> Utilise Mersenne Twister (Standard)
// Si type==1 -> Utilise Xorshift (Fait main, rapide)
double random_01(int type) {
    if (type == 0) {
        // Méthode classique C++
        uniform_real_distribution<> dist(0.0, 1.0);
        return dist(generateur_mt);
    } else {
        // Méthode Xorshift (calcul binaire rapide)
        unsigned int x = graine_xor;
        x ^= x << 13;
        x ^= x >> 17;
        x ^= x << 5;
        graine_xor = x;
        // On divise pour ramener entre 0 et 1
        return (double)x / 4294967295.0;
    }
}

// Fonction qui renvoie un entier entre 0 et max
int random_int(int type, int max) {
    if (type == 0) {
        uniform_int_distribution<> dist(0, max - 1);
        return dist(generateur_mt);
    } else {
        unsigned int x = graine_xor;
        x ^= x << 13; x ^= x >> 17; x ^= x << 5;
        graine_xor = x;
        return x % max;
    }
}

// Tirage loi exponentielle
int expo(int type, double moy) {
    double u = random_01(type);
    if(u >= 1.0) u = 0.999999;
    return (int)round(-moy * log(1.0 - u));
}

int main(int argc, char* argv[]) {
    // Lecture des arguments : ./prog [GRAINE] [TYPE]
    int graine = 1;
    int type = 0; // 0=MT, 1=Xorshift

    if(argc > 1) graine = atoi(argv[1]);
    if(argc > 2) type = atoi(argv[2]);

    init_random(graine);

    // 1. On crée la population
    vector<Humain> pop(N);
    for(int i=0; i<N; i++) {
        pop[i].x = random_int(type, GRILLE);
        pop[i].y = random_int(type, GRILLE);
        pop[i].etat = S;
        pop[i].compteur = 0;
        pop[i].dE = expo(type, 3.0);
        pop[i].dI = expo(type, 7.0);
        pop[i].dR = expo(type, 365.0);
    }

    // On infecte les 20 premiers
    for(int i=0; i<20; i++) pop[i].etat = I;

    // Fichier de sortie
    string nom = (type == 0) ? "MT" : "XOR";
    string fichier_nom = "bonus_" + nom + "_" + to_string(graine) + ".csv";
    ofstream f(fichier_nom);
    f << "jour,I\n"; // On sauvegarde juste le nombre d'infectés

    // 2. Simulation
    for(int j=0; j<JOURS; j++) {

        // On vide la grille
        static vector<int> grille[GRILLE*GRILLE];
        for(int k=0; k<GRILLE*GRILLE; k++) grille[k].clear();

        // Déplacement des agents
        for(int i=0; i<N; i++) {
            pop[i].x = random_int(type, GRILLE);
            pop[i].y = random_int(type, GRILLE);
            int case_idx = pop[i].x * GRILLE + pop[i].y;
            grille[case_idx].push_back(i);
        }

        int nb_I = 0;

        // Boucle sur chaque humain
        for(int i=0; i<N; i++) {

            // Si Susceptible -> Peut devenir Infecté
            if(pop[i].etat == S) {
                int n_voisins_I = 0;

                // On regarde les 8 cases autour
                for(int dx=-1; dx<=1; dx++) {
                    for(int dy=-1; dy<=1; dy++) {
                        int vx = (pop[i].x + dx + GRILLE) % GRILLE;
                        int vy = (pop[i].y + dy + GRILLE) % GRILLE;
                        int v_idx = vx * GRILLE + vy;

                        // Qui est là ?
                        for(int id : grille[v_idx]) {
                            if(pop[id].etat == I) n_voisins_I++;
                        }
                    }
                }

                // Infection ?
                if(n_voisins_I > 0) {
                    double p = 1.0 - exp(-0.5 * n_voisins_I);
                    if(random_01(type) < p) {
                        pop[i].etat = E;
                        pop[i].compteur = 0;
                    }
                }
            }
            // Changements d'états (Automate simple)
            else if(pop[i].etat == E) {
                if(pop[i].compteur >= pop[i].dE) { pop[i].etat=I; pop[i].compteur=0; }
            }
            else if(pop[i].etat == I) {
                if(pop[i].compteur >= pop[i].dI) { pop[i].etat=R; pop[i].compteur=0; }
            }
            else if(pop[i].etat == R) {
                if(pop[i].compteur >= pop[i].dR) { pop[i].etat=S; pop[i].compteur=0; }
            }

            pop[i].compteur++;
            if(pop[i].etat == I) nb_I++;
        }

        // On écrit le résultat du jour
        f << j << "," << nb_I << "\n";
    }

    f.close();
    return 0;
}



Writing bonus_etudiant.cpp


In [ ]:
%%bash
echo "1. Compilation du programme..."
g++ -O3 bonus_etudiant.cpp -o bonus_exe

echo "2. Lancement des simulations Mersenne Twister (Standard)..."
# On lance 5 fois avec des graines différentes (1, 2, 3, 4, 5)
for i in 1 2 3 4 5; do
    ./bonus_exe $i 0
done

echo "3. Lancement des simulations Xorshift (Rapide)..."
# On lance 5 fois aussi
for i in 1 2 3 4 5; do
    ./bonus_exe $i 1
done

echo "Terminé ! Les fichiers sont prêts."



1. Compilation du programme...
2. Lancement des simulations Mersenne Twister (Standard)...
3. Lancement des simulations Xorshift (Rapide)...
Terminé ! Les fichiers sont prêts.


In [ ]:
import pandas as pd
import glob
import numpy as np

# Fonction simple pour calculer la moyenne du pic d'infectés
def calculer_moyenne(motif_fichier):
    liste_fichiers = glob.glob(motif_fichier)
    pics = []
    for fichier in liste_fichiers:
        data = pd.read_csv(fichier)
        pic_max = data['I'].max() # Le maximum d'infectés de cette simulation
        pics.append(pic_max)
    return np.mean(pics)

# On calcule pour les deux méthodes
moyenne_mt = calculer_moyenne("bonus_MT_*.csv")
moyenne_xor = calculer_moyenne("bonus_XOR_*.csv")

print(f"Pic moyen avec Mersenne Twister (Standard) : {moyenne_mt:.0f} personnes")
print(f"Pic moyen avec Xorshift (Rapide)         : {moyenne_xor:.0f} personnes")

# Petite conclusion automatique
diff = abs(moyenne_mt - moyenne_xor)
print(f"Différence : {diff:.0f}")

if diff < 200:
    print("\n>>> CONCLUSION : La différence est minime.")
    print(">>> On peut utiliser le générateur rapide sans fausser la simulation.")
    print(">>> BONUS VALIDÉ ")
else:
    print("\n>>> CONCLUSION : Il y a une différence notable.")



Pic moyen avec Mersenne Twister (Standard) : 6976 personnes
Pic moyen avec Xorshift (Rapide)         : 6861 personnes
Différence : 115

>>> CONCLUSION : La différence est minime.
>>> On peut utiliser le générateur rapide sans fausser la simulation.
>>> BONUS VALIDÉ ✅


In [ ]:
%%bash
# On mesure le temps pour 50 simulations (pour bien voir la différence)
g++ -O3 bonus_etudiant.cpp -o bonus_exe

echo "Mesure de vitesse Mersenne Twister (50 runs)..."
time for i in {1..50}; do ./bonus_exe $i 0 > /dev/null; done

echo -e "\nMesure de vitesse Xorshift (50 runs)..."
time for i in {1..50}; do ./bonus_exe $i 1 > /dev/null; done


Mesure de vitesse Mersenne Twister (50 runs)...

Mesure de vitesse Xorshift (50 runs)...



real	1m37.251s
user	1m35.818s
sys	0m0.323s

real	0m57.419s
user	0m56.754s
sys	0m0.269s
